# Creating and Using Catalogs

Catalogs for datasets and model output are made using `intake`. This page demos how to make your own model output catalog; data will be separate, although many are currently available.

In [1]:
import ocean_skill as osk
import ocean_skill
from ocean_skill.build import build_kerchunk, build_catalog, discover_opendap_files
from ocean_skill.obs.modis import nickname
from pathlib import Path


## Catalog heirarchy

| Priority | Location | How to set it up | Defaults to | Notes |
|---|---|---|---|---|
| 1 (lowest) | Packaged — `ocean_skill/catalogs/` | Ships with the package install | 8 reference catalogs (WOA, GLODAP, MODIS Aqua, etc.) | Now actually included in built wheels — was a packaging bug |
| 2 | Shared/team | `$OCEAN_SKILL_CATALOGS` env var (`os.pathsep`-separated dirs), or `osk.catalog.add_search_path(dir)` in code | Nothing | For a cluster module file, team `.bashrc`, or a notebook registering a group directory |
| 3 | User | Drop YAML files in `~/.ocean-skill/catalogs/` | Nothing | Same path on macOS/Linux/Windows; the old hidden `platformdirs` location is still scanned as a fallback just below it |
| 4 (highest) | Project-local | `./catalogs/` in your project (auto-discovered, gitignored) | Nothing | Unchanged; wins because it's the narrowest scope |

Check what's actually active any time with:

In [3]:
osk.catalog.search_paths()

[PosixPath('/mnt/cooltub/cw/user/kthyng/projects/ocean-skill/ocean_skill/catalogs'),
 PosixPath('/home/kthyng/.config/ocean-skill/catalogs'),
 PosixPath('/mnt/cooltub/cw/user/kthyng/projects/ocean-skill/catalogs'),
 PosixPath('/mnt/cooltub/cw/user/kthyng/projects/ocean-skill/docs/catalogs')]

### Point a shared catalog at pre-staged local data (e.g. GLODAP on Anvil)

Some sources are catalogued as a *build recipe*, not a bare URL — GLODAP is a NOAA
`.tar.gz` of one NetCDF per variable, read by `ocean_skill.readers.PoochTarNetCDF`,
which downloads + untars it. On a cluster where that data is already staged on a
shared filesystem, point the same reader at the local directory instead with
`local_dir=` — everything downstream (the per-variable merge that avoids GLODAP's
conflicting-diagnostics problem) is identical, so the resulting `glodap` Dataset is
the same either way, only the byte source differs.

Build this once and save it into a directory on `$OCEAN_SKILL_CATALOGS` (the shared
tier) so it shadows the packaged, internet-backed `glodap` entry for the whole group —
`osk.read("glodap")` then reads the local copy without anyone else changing anything.

In [ ]:
from ocean_skill.build import build_catalog

# Same name_map GLODAP's shipped catalog uses, so standard_names/variables come out
# identical to the internet-backed entry (GLODAP's raw files carry no standard_name
# attrs of their own).
GLODAP_NAME_MAP = {
    "NO3": "mole_concentration_of_nitrate_in_sea_water",
    "PO4": "mole_concentration_of_phosphate_in_sea_water",
    "TAlk": "sea_water_alkalinity_expressed_as_mole_equivalent",
    "TCO2": "mole_concentration_of_dissolved_inorganic_carbon_in_sea_water",
    "oxygen": "mole_concentration_of_dissolved_molecular_oxygen_in_sea_water",
    "salinity": "sea_water_practical_salinity",
    "silicate": "mole_concentration_of_silicate_in_sea_water",
    "temperature": "sea_water_potential_temperature",
}

SHARED = "/anvil/projects/x-ees250129/cstar-forge-data/catalogs"  # a dir on $OCEAN_SKILL_CATALOGS

build_catalog(
    {
        "glodap": {
            "reader": "ocean_skill.readers:PoochTarNetCDF",
            "reader_kwargs": {
                # same reader as the internet entry, just local_dir instead of url
                "local_dir": "/anvil/projects/x-ees250129/cstar-forge-data/"
                "source-data/GLODAP/GLODAPv2.2016b",
                "member_glob": "*.nc",
                "var_from_filename": True,
                "keep_vars": ["Depth"],
            },
            "name_map": GLODAP_NAME_MAP,
            # carried over from the shipped catalog so osk.find()/osk.describe() see
            # the same descriptive metadata either way
            "institution": "NOAA NCEI / GLODAPv2",
            "climatology": True,
            "depth_coord": "depth_surface",
            "nominal_resolution_km": 111,
            "notes": "Open-ocean product: marginal seas are largely masked (~8% "
            "coverage in the Gulf of Mexico).",
            "references": "Olsen et al. 2016 (ESSD); Lauvset et al. GLODAPv2.2016b",
        }
    },
    f"{SHARED}/glodap.catalog.yaml",
    title="GLODAPv2.2016b Mapped Climatology",
)

## Vocabulary handling

In [ ]:
FILL IN

## Set up catalog for model output

`out_dir` is where the kerchunk parquet file is stored for the model output, which is then referenced by the catalog as the model location.

### Iceland

In [ ]:
%%time

# this took 8 min on Anvil

grid = "/anvil/projects/x-ees250129/x-uheede/Iceland_experiments/Iceland2_MARBL_2024_60m/P_INPUT/Iceland2_grid.nc"
base = "/anvil/projects/x-ees250129/x-uheede/Iceland_experiments/Iceland2_MARBL_2024_60m"
models = {"his": "Iceland2_MARBL_2024_his.*.nc", "bgc": "Iceland2_MARBL_2024_bgc.*.nc", "bgc_dia": "Iceland2_MARBL_2024_bgc_dia.*.nc"}


refs1 = build_kerchunk({"his": models["his"]}, root=base, grid=grid,
                      out_dir="/home/x-kthyng/projects/ocean-skill/localfiles/")

### Two models on the same grid

`refs1` sets up multiple sets of model output for the same grid and `refs2` sets up another two sets of model output. These are then combined into one catalog to represent all of the models.

In [2]:
%%time

refs1 = build_kerchunk({"dev": "cson_roms-marbl_v0.1_Gulf_of_Alaska_64procs_20100101-20100701_dev/joined_output/*.nc", 
                       "dev_marblsub": "cson_roms-marbl_v0.1_Gulf_of_Alaska_64procs_20100101-20100701_dev_marblsub/joined_output/*.nc"},
                      root="/mnt/scratch/saved_runs/", 
                      grid="/mnt/scratch/cstar-forge-data/cson_roms-marbl_v0.1_Gulf_of_Alaska_64procs/input_data/cson_roms-marbl_v0_1_Gulf_of_Alaska_64procs_grid.nc", 
                      keep="latest-per-file",  # use this for restart files to keep the main time step
                      out_dir="/home/kthyng/projects/ocean-skill/localfiles/")

refs2 = build_kerchunk({"ccs": "output_rst.*.nc", 
                       "ccs_cdr": "output_cdr.??????????????.nc"},
                      root="/mnt/cooltub/cw/user/blsaenz/cstar-roms-run/cson_roms-marbl_v0.1_ccs-4km_128procs_128cores/joined_output/", 
                      grid="/mnt/cooltub/cw/user/blsaenz/cstar-roms-run/cson_roms-marbl_v0.1_ccs-4km_128procs_128cores/input/input_datasets/cson_roms-marbl_v0_1_ccs-4km_128procs_grid.nc", 
                      keep="latest-per-file",  # use this for restart files to keep the main time step
                      out_dir="/home/kthyng/projects/ocean-skill/localfiles/")


CPU times: user 19.5 s, sys: 978 ms, total: 20.5 s
Wall time: 25.8 s


In [4]:
build_catalog(refs1 | refs2, "catalogs/ben_dev_tanagra.yaml", title="Bens comparison runs on tanagra")

PosixPath('catalogs/ben_dev_tanagra.yaml')

## Using Catalogs

### Available catalogs

In [9]:
osk.catalogs.catalog_names()

['Bens comparison runs on tanagra',
 'CalOOS TimeSeries Datasets',
 'Copernicus Marine',
 'GLODAPv2.2016b Mapped Climatologies',
 'Global mixed layer depth climatologies',
 'MODIS Aqua',
 'NOAA CoastWatch',
 'OOI Station Papa',
 'OceanSODA',
 'WHOTS mooring (OceanSITES)',
 'World Ocean Atlas 2023 (1 deg) — annual + monthly climatology']

### Look at available data sources

`osk.catalogs` or `list(osk.catalogs)` or `osk.catalogs.names()`

In [10]:
list(osk.catalogs)

['NOAA_DHW',
 'NWW3_Global_Best',
 'erdMBsstd1day',
 'erdMH1cflh1day_R2022NRT',
 'erdMH1cflh1day_R2022SQ',
 'erdMH1chla1day_R2022NRT',
 'erdMH1chla1day_R2022SQ',
 'erdMH1sstd1day_R2022NRTMasked',
 'erdMH1sstd1day_R2022SQMasked',
 'erdMPIC1day_R2022NRT',
 'erdMPIC1day_R2022SQ',
 'erdMPOC1day_R2022NRT',
 'erdMPOC1day_R2022SQ',
 'erdMWchla1day',
 'erdMWsstd1day',
 'erdQCwindproducts1day',
 'jplMURSST41',
 'jplMURSST41anom1day',
 'jplMURSST41clim',
 'jplMURSST41mday',
 'jplMURSST42',
 'ncdcOisst21Agg',
 'nceiErsstv5',
 'nesdisVHNsstDaily',
 'noaacwNPPN20S3ASCIDINEOF2kmDaily',
 'noaacwNPPN20S3AkdSCIDINEOF2kmDaily',
 'noaacwNPPN20S3AspmSCIDINEOF2kmDaily',
 'nsidcG02202v6nh1day',
 'nsidcG02202v6sh1day',
 'productivity_viirs_snpp_daily',
 'productivity_viirs_snpp_nrt_daily',
 'chl_climatology_doy_geo',
 'chl_climatology_doy_timeseries',
 'chl_gapfree_my_daily_geo',
 'chl_gapfree_my_daily_timeseries',
 'chl_gapfree_nrt_daily_geo',
 'chl_gapfree_nrt_daily_timeseries',
 'cur_multiobs_my_daily_geo

### List sources in one catalog

In [15]:
osk.describe('NOAA CoastWatch')

catalog: NOAA CoastWatch
  featureTypes: ['grid']
  geospatial_lat_max: 5837500.0
  geospatial_lat_min: -5337500.0
  geospatial_lon_max: 3937500.0
  geospatial_lon_min: -3937500.0
  geospatial_vertical_max: 10.0
  geospatial_vertical_min: 0.0
  standard_names: ['concentration_of_chlorophyll_in_sea_water', 'diffuse_attenuation_coefficient_of_downwelling_radiative_flux_in_sea_water', 'divergence_of_wind', 'eastward_wind', 'land_binary_mask', 'mass_concentration_of_chlorophyll_a_in_sea_water', 'mass_concentration_of_chlorophyll_in_sea_water', 'net_primary_productivity_of_carbon', 'northward_wind', 'sea_ice_area_fraction', 'sea_surface_foundation_temperature', 'sea_surface_foundation_temperature_anomaly', 'sea_surface_swell_wave_from_direction', 'sea_surface_swell_wave_period', 'sea_surface_swell_wave_significant_height', 'sea_surface_temperature', 'sea_surface_wave_from_direction_at_variance_spectral_density_maximum', 'sea_surface_wave_period_at_variance_spectral_density_maximum', 'sea_surface_wave_significant_height', 'sea_surface_wind_wave_from_direction', 'sea_surface_wind_wave_period', 'sea_surface_wind_wave_significant_height', 'status_flag', 'surface_downward_eastward_stress', 'surface_downward_northward_stress', 'surface_downwelling_photosynthetic_photon_flux_in_air', 'surface_temperature_anomaly', 'suspended particulate matter', 'wind_from_direction', 'wind_speed']
  time_coverage_end: 2026-08-19
  time_coverage_start: 1854-01-01
  title: NOAA CoastWatch
  sources (31): NOAA_DHW, NWW3_Global_Best, erdMBsstd1day, erdMH1cflh1day_R2022NRT, erdMH1cflh1day_R2022SQ, erdMH1chla1day_R2022NRT, erdMH1chla1day_R2022SQ, erdMH1sstd1day_R2022NRTMasked, erdMH1sstd1day_R2022SQMasked, erdMPIC1day_R2022NRT, erdMPIC1day_R2022SQ, erdMPOC1day_R2022NRT, erdMPOC1day_R2022SQ, erdMWchla1day, erdMWsstd1day, erdQCwindproducts1day, jplMURSST41, jplMURSST41anom1day, jplMURSST41clim, jplMURSST41mday, jplMURSST42, ncdcOisst21Agg, nceiErsstv5, nesdisVHNsstDaily, noaacwNPPN20S3ASCIDINEOF2kmDaily, noaacwNPPN20S3AkdSCIDINEOF2kmDaily, noaacwNPPN20S3AspmSCIDINEOF2kmDaily, nsidcG02202v6nh1day, nsidcG02202v6sh1day, productivity_viirs_snpp_daily, productivity_viirs_snpp_nrt_daily
  vocabulary:
    matched (7):
      chlorophyll     <- concentration_of_chlorophyll_in_sea_water, mass_concentration_of_chlorophyll_a_in_sea_water, mass_concentration_of_chlorophyll_in_sea_water   (3 variables)
      eastward_wind   <- eastward_wind
      kd490           <- diffuse_attenuation_coefficient_of_downwelling_radiative_flux_in_sea_water
      northward_wind  <- northward_wind
      sea_ice         <- sea_ice_area_fraction
      temperature     <- sea_surface_foundation_temperature, sea_surface_temperature   (2 variables)
      wind_speed      <- wind_speed
    unmatched (20):
      divergence_of_wind, land_binary_mask, net_primary_productivity_of_carbon, sea_surface_foundation_temperature_anomaly, sea_surface_swell_wave_from_direction, sea_surface_swell_wave_period, sea_surface_swell_wave_significant_height, sea_surface_wave_from_direction_at_variance_spectral_density_maximum, sea_surface_wave_period_at_variance_spectral_density_maximum, sea_surface_wave_significant_height, sea_surface_wind_wave_from_direction, sea_surface_wind_wave_period, sea_surface_wind_wave_significant_height, status_flag, surface_downward_eastward_stress, surface_downward_northward_stress, surface_downwelling_photosynthetic_photon_flux_in_air, surface_temperature_anomaly, suspended particulate matter, wind_from_direction
  coordinates:
    matched (4):
      T (time)       <- time
      X (longitude)  <- longitude
      Y (latitude)   <- latitude
      Z (vertical)   <- depth
    missing (0)

### Search across catalogs

Show a bunch of search terms to use

In [16]:
osk.find?

Signature:
osk.find(
    *,
    text: 'str | list[str] | None' = None,
    name: 'str | None' = None,
    catalog: 'str | None' = None,
    climatology: 'bool | str | None' = None,
    variable: 'str | None' = None,
    featureType: 'str | None' = None,
    bbox: 'tuple[float, float, float, float] | None' = None,
    time: 'tuple[str, str] | None' = None,
    resolution: 'float | tuple[float | None, float | None] | None' = None,
    cadence: 'str | float | tuple[float | None, float | None] | None' = None,
    vertical: 'bool | None' = None,
) -> 'SourceNames'
Docstring:
Search discovered sources by name and metadata; return matching source names.

Only the filters given are applied::

    osk.find(text="modis chl jan")               # free text, all terms must match
    osk.find(variable="nitrate")                 # any spelling of the variable
    osk.find(name="papa")                        # substring, case-insensitive
    osk.find(name="woa23_nitrate_month*")        # glob
    osk.

In [27]:
osk.find(climatology="January", resolution=5)

['woa23_nitrate_month01',
 'woa23_oxygen_month01',
 'woa23_phosphate_month01',
 'woa23_salinity_month01',
 'woa23_silicate_month01',
 'woa23_temperature_month01']

In [24]:
osk.find(cadence="daily", vertical=True)

['cur_multiobs_my_daily_geo',
 'cur_multiobs_my_daily_timeseries',
 'cur_multiobs_nrt_daily_geo',
 'cur_multiobs_nrt_daily_timeseries',
 'glorys_my_daily_geo',
 'glorys_my_daily_timeseries']

In [28]:
osk.find(catalog="soda")

['oceansoda']

In [22]:
osk.find(name="papa", featureType="timeSeries")

['ooi-gp02hypm-rim01-02-ctdmog039',
 'ooi-gp02hypm-wfp02-01-flordl000',
 'ooi-gp02hypm-wfp02-03-dostal000',
 'ooi-gp02hypm-wfp02-04-ctdpfl000',
 'ooi-gp02hypm-wfp02-05-vel3dl000',
 'ooi-gp02hypm-wfp03-01-flordl000',
 'ooi-gp02hypm-wfp03-03-dostal000',
 'ooi-gp02hypm-wfp03-04-ctdpfl000',
 'ooi-gp02hypm-wfp03-05-vel3dl000',
 'ooi-gp03flma-rim01-02-adcpsl003',
 'ooi-gp03flma-rim01-02-ctdmog040',
 'ooi-gp03flma-rim01-02-ctdmog041',
 'ooi-gp03flma-rim01-02-ctdmog042',
 'ooi-gp03flma-rim01-02-ctdmog043',
 'ooi-gp03flma-rim01-02-ctdmog044',
 'ooi-gp03flma-rim01-02-ctdmog045',
 'ooi-gp03flma-rim01-02-ctdmog046',
 'ooi-gp03flma-rim01-02-ctdmog047',
 'ooi-gp03flma-rim01-02-ctdmog048',
 'ooi-gp03flma-rim01-02-ctdmoh049',
 'ooi-gp03flma-rim01-02-ctdmoh050',
 'ooi-gp03flma-rim01-02-ctdmoh051',
 'ooi-gp03flma-ris01-03-dostad000',
 'ooi-gp03flma-ris01-04-phsenf000',
 'ooi-gp03flma-ris01-05-flortd000',
 'ooi-gp03flmb-rim01-02-adcpsl007',
 'ooi-gp03flmb-rim01-02-ctdmog060',
 'ooi-gp03flmb-rim01-02-ctdm

In [19]:
osk.find(variable="temperature")

['NOAA_DHW',
 'erdMBsstd1day',
 'erdMH1sstd1day_R2022NRTMasked',
 'erdMH1sstd1day_R2022SQMasked',
 'erdMWsstd1day',
 'jplMURSST41',
 'jplMURSST41clim',
 'jplMURSST41mday',
 'jplMURSST42',
 'ncdcOisst21Agg',
 'nceiErsstv5',
 'nesdisVHNsstDaily',
 'productivity_viirs_snpp_daily',
 'productivity_viirs_snpp_nrt_daily',
 'glorys_climatology_geo',
 'glorys_climatology_timeseries',
 'glorys_my_daily_geo',
 'glorys_my_daily_timeseries',
 'sst_c3s_rep_daily_geo',
 'sst_c3s_rep_daily_timeseries',
 'sst_esacci_rep_daily_geo',
 'sst_esacci_rep_daily_timeseries',
 'sst_ostia_nrt_daily_geo',
 'sst_ostia_nrt_daily_timeseries',
 'glodap',
 'oceansoda',
 'ooi-gp02hypm-rim01-02-ctdmog039',
 'ooi-gp03flma-rim01-02-ctdmog040',
 'ooi-gp03flma-rim01-02-ctdmog041',
 'ooi-gp03flma-rim01-02-ctdmog042',
 'ooi-gp03flma-rim01-02-ctdmog043',
 'ooi-gp03flma-rim01-02-ctdmog044',
 'ooi-gp03flma-rim01-02-ctdmog045',
 'ooi-gp03flma-rim01-02-ctdmog046',
 'ooi-gp03flma-rim01-02-ctdmog047',
 'ooi-gp03flma-rim01-02-ctdmog0

### Examine metadata in a catalog

In [32]:
osk.describe('NOAA CoastWatch')

catalog: NOAA CoastWatch
  featureTypes: ['grid']
  geospatial_lat_max: 5837500.0
  geospatial_lat_min: -5337500.0
  geospatial_lon_max: 3937500.0
  geospatial_lon_min: -3937500.0
  geospatial_vertical_max: 10.0
  geospatial_vertical_min: 0.0
  standard_names: ['concentration_of_chlorophyll_in_sea_water', 'diffuse_attenuation_coefficient_of_downwelling_radiative_flux_in_sea_water', 'divergence_of_wind', 'eastward_wind', 'land_binary_mask', 'mass_concentration_of_chlorophyll_a_in_sea_water', 'mass_concentration_of_chlorophyll_in_sea_water', 'net_primary_productivity_of_carbon', 'northward_wind', 'sea_ice_area_fraction', 'sea_surface_foundation_temperature', 'sea_surface_foundation_temperature_anomaly', 'sea_surface_swell_wave_from_direction', 'sea_surface_swell_wave_period', 'sea_surface_swell_wave_significant_height', 'sea_surface_temperature', 'sea_surface_wave_from_direction_at_variance_spectral_density_maximum', 'sea_surface_wave_period_at_variance_spectral_density_maximum', 'sea_surface_wave_significant_height', 'sea_surface_wind_wave_from_direction', 'sea_surface_wind_wave_period', 'sea_surface_wind_wave_significant_height', 'status_flag', 'surface_downward_eastward_stress', 'surface_downward_northward_stress', 'surface_downwelling_photosynthetic_photon_flux_in_air', 'surface_temperature_anomaly', 'suspended particulate matter', 'wind_from_direction', 'wind_speed']
  time_coverage_end: 2026-08-19
  time_coverage_start: 1854-01-01
  title: NOAA CoastWatch
  sources (31): NOAA_DHW, NWW3_Global_Best, erdMBsstd1day, erdMH1cflh1day_R2022NRT, erdMH1cflh1day_R2022SQ, erdMH1chla1day_R2022NRT, erdMH1chla1day_R2022SQ, erdMH1sstd1day_R2022NRTMasked, erdMH1sstd1day_R2022SQMasked, erdMPIC1day_R2022NRT, erdMPIC1day_R2022SQ, erdMPOC1day_R2022NRT, erdMPOC1day_R2022SQ, erdMWchla1day, erdMWsstd1day, erdQCwindproducts1day, jplMURSST41, jplMURSST41anom1day, jplMURSST41clim, jplMURSST41mday, jplMURSST42, ncdcOisst21Agg, nceiErsstv5, nesdisVHNsstDaily, noaacwNPPN20S3ASCIDINEOF2kmDaily, noaacwNPPN20S3AkdSCIDINEOF2kmDaily, noaacwNPPN20S3AspmSCIDINEOF2kmDaily, nsidcG02202v6nh1day, nsidcG02202v6sh1day, productivity_viirs_snpp_daily, productivity_viirs_snpp_nrt_daily
  vocabulary:
    matched (7):
      chlorophyll     <- concentration_of_chlorophyll_in_sea_water, mass_concentration_of_chlorophyll_a_in_sea_water, mass_concentration_of_chlorophyll_in_sea_water   (3 variables)
      eastward_wind   <- eastward_wind
      kd490           <- diffuse_attenuation_coefficient_of_downwelling_radiative_flux_in_sea_water
      northward_wind  <- northward_wind
      sea_ice         <- sea_ice_area_fraction
      temperature     <- sea_surface_foundation_temperature, sea_surface_temperature   (2 variables)
      wind_speed      <- wind_speed
    unmatched (20):
      divergence_of_wind, land_binary_mask, net_primary_productivity_of_carbon, sea_surface_foundation_temperature_anomaly, sea_surface_swell_wave_from_direction, sea_surface_swell_wave_period, sea_surface_swell_wave_significant_height, sea_surface_wave_from_direction_at_variance_spectral_density_maximum, sea_surface_wave_period_at_variance_spectral_density_maximum, sea_surface_wave_significant_height, sea_surface_wind_wave_from_direction, sea_surface_wind_wave_period, sea_surface_wind_wave_significant_height, status_flag, surface_downward_eastward_stress, surface_downward_northward_stress, surface_downwelling_photosynthetic_photon_flux_in_air, surface_temperature_anomaly, suspended particulate matter, wind_from_direction
  coordinates:
    matched (4):
      T (time)       <- time
      X (longitude)  <- longitude
      Y (latitude)   <- latitude
      Z (vertical)   <- depth
    missing (0)

### Examine metadata in a source within a catalog

Note that matched variables and coordinates are indicated at the bottom according to the active vocabulary at the moment (this is not saved into the catalog).

In [33]:
osk.describe("NOAA_DHW")

source: NOAA CoastWatch:NOAA_DHW
  path: /mnt/cooltub/cw/user/kthyng/projects/ocean-skill/ocean_skill/catalogs/coastwatch.yaml
  axes: {'T': 'time', 'X': 'longitude', 'Y': 'latitude'}
  featureType: grid
  featureType_source: inferred
  geospatial_lat_max: 89.9749984741211
  geospatial_lat_min: -89.9749984741211
  geospatial_lon_max: 179.97500610351562
  geospatial_lon_min: -179.97500610351562
  grid_regular: True
  grid_resolution_deg: 0.049999
  grid_resolution_km: 5.56
  lon_convention: -180-180
  standard_names: {'CRW_SEAICE': 'sea_ice_area_fraction', 'CRW_SST': 'sea_surface_temperature'}
  time_coverage_end: 2026-08-12
  time_coverage_start: 1985-04-01
  time_resolution: P1D
  time_resolution_s: 86400.0
  variables: ['sea_ice_area_fraction', 'sea_surface_temperature']
  vocabulary:
    matched (2):
      sea_ice      <- sea_ice_area_fraction
      temperature  <- sea_surface_temperature
    unmatched (0)
  coordinates:
    matched (3):
      T (time)       <- time
      X (longitude)  <- longitude
      Y (latitude)   <- latitude
    missing (1): Z

### Read in source

`osk.read(SOURCE_NAME)`

Note that many time series sources are backed by ERDDAP servers so they can be queried using ERDDAP constraints, which can greatly speed up their read in timing if you limit the amount of time read in. Many gridded datasets are lazily read in without any special inputs.

In [46]:
osk.read("NOAA_DHW")

<xarray.Dataset> Size: 27TB
Dimensions:                  (time: 15122, latitude: 3600, longitude: 7200)
Coordinates:
  * time                     (time) datetime64[ns] 121kB 1985-04-01T12:00:00 ...
  * latitude                 (latitude) float32 14kB 89.97 89.93 ... -89.97
  * longitude                (longitude) float32 29kB -180.0 -179.9 ... 180.0
Data variables:
    CRW_BAA                  (time, latitude, longitude) float32 2TB dask.array<chunksize=(667, 158, 317), meta=np.ndarray>
    CRW_BAA_mask             (time, latitude, longitude) float32 2TB dask.array<chunksize=(667, 158, 317), meta=np.ndarray>
    CRW_BAA_7D_MAX           (time, latitude, longitude) float32 2TB dask.array<chunksize=(667, 158, 317), meta=np.ndarray>
    CRW_BAA_7D_MAX_mask      (time, latitude, longitude) float32 2TB dask.array<chunksize=(667, 158, 317), meta=np.ndarray>
    CRW_DHW                  (time, latitude, longitude) float64 3TB dask.array<chunksize=(530, 125, 252), meta=np.ndarray>
    CRW_DHW_mask             (time, latitude, longitude) float32 2TB dask.array<chunksize=(667, 158, 317), meta=np.ndarray>
    CRW_HOTSPOT              (time, latitude, longitude) float64 3TB dask.array<chunksize=(530, 125, 252), meta=np.ndarray>
    CRW_HOTSPOT_mask         (time, latitude, longitude) float32 2TB dask.array<chunksize=(667, 158, 317), meta=np.ndarray>
    sea_ice_area_fraction    (time, latitude, longitude) float64 3TB dask.array<chunksize=(530, 125, 252), meta=np.ndarray>
    sea_surface_temperature  (time, latitude, longitude) float64 3TB dask.array<chunksize=(530, 125, 252), meta=np.ndarray>
    CRW_SSTANOMALY           (time, latitude, longitude) float64 3TB dask.array<chunksize=(530, 125, 252), meta=np.ndarray>
    CRW_SSTANOMALY_mask      (time, latitude, longitude) float32 2TB dask.array<chunksize=(667, 158, 317), meta=np.ndarray>
Attributes: (12/63)
    acknowledgement:            NOAA Coral Reef Watch (CRW)
    cdm_data_type:              Grid
    comment:                    This product is designed to improve on and re...
    Conventions:                CF-1.6, ACDD-1.3
    creator_email:              coralreefwatch@noaa.gov
    creator_institution:        NOAA Coral Reef Watch (CRW)
    ...                         ...
    time_coverage_duration:     P1D
    time_coverage_end:          2026-09-01T12:00:00Z
    time_coverage_resolution:   P1D
    time_coverage_start:        1985-04-01T12:00:00Z
    title:                      NOAA Coral Reef Watch Operational Daily Near-...
    Westernmost_Easting:        -179.975

In [50]:
osk.read('ooi-gp02hypm-rim01-02-ctdmog039', constraints={"time>=": "2025-01-01"})

,time (UTC),latitude (degrees_north),longitude (degrees_east),z (m),sea_water_electrical_conductivity,sea_water_electrical_conductivity_qc_agg,sea_water_electrical_conductivity_qc_tests,depth_reading,depth_reading_qc_agg,depth_reading_qc_tests,...,sea_water_density,sea_water_density_qc_agg,sea_water_density_qc_tests,sea_water_pressure,sea_water_pressure_qc_agg,sea_water_pressure_qc_tests,sea_water_temperature,sea_water_temperature_qc_agg,sea_water_temperature_qc_tests,station
0,2025-01-01 00:00:00+00:00,50.069483,-144.8022,0.0,33.067903,1,NaN,154.683246,2,NaN,...,1027.283527,2,NaN,156.081226,1,NaN,5.755385,1,NaN,NaN
1,2025-01-01 00:15:00+00:00,50.069483,-144.8022,0.0,33.023828,1,NaN,154.757224,2,NaN,...,1027.310578,2,NaN,156.155901,1,NaN,5.683047,1,NaN,NaN
2,2025-01-01 00:30:00+00:00,50.069483,-144.8022,0.0,32.908817,1,NaN,154.471591,2,NaN,...,1027.309873,2,NaN,155.867578,1,NaN,5.569774,1,NaN,NaN
3,2025-01-01 00:45:00+00:00,50.069483,-144.8022,0.0,33.008620,1,NaN,154.406691,2,NaN,...,1027.293988,2,NaN,155.802066,1,NaN,5.684662,1,NaN,NaN
4,2025-01-01 01:00:00+00:00,50.069483,-144.8022,0.0,33.045950,1,NaN,154.265036,2,NaN,...,1027.286557,2,NaN,155.659077,1,NaN,5.728645,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14012,2025-08-21 16:00:00+00:00,50.069483,-144.8022,0.0,32.616100,1,NaN,161.631699,2,NaN,...,1027.386575,2,NaN,163.094582,1,NaN,5.231100,1,NaN,NaN
14013,2025-08-22 00:00:00+00:00,50.069483,-144.8022,0.0,32.584400,1,NaN,162.916393,2,NaN,...,1027.382133,2,NaN,164.391420,1,NaN,5.210600,1,NaN,NaN
14014,2025-08-22 08:00:00+00:00,50.069483,-144.8022,0.0,32.546700,1,NaN,163.937104,2,NaN,...,1027.420386,2,NaN,165.421784,1,NaN,5.136300,1,NaN,NaN
14015,2025-08-24 04:00:00+00:00,50.069483,-144.8022,0.0,32.604900,1,NaN,162.089263,2,NaN,...,1027.369964,2,NaN,163.556470,1,NaN,5.240300,1,NaN,NaN


In [52]:
osk.read("noaa_nos_co_ops_9414746", constraints={"time>=": "2026-01-01"})

,time (UTC),latitude (degrees_north),longitude (degrees_east),z (m),sea_surface_height_amplitude_due_to_geocentric_ocean_tide_above_mllw,sea_surface_height_amplitude_due_to_geocentric_ocean_tide_above_mllw_qc_agg,sea_surface_height_amplitude_due_to_geocentric_ocean_tide_above_mllw_qc_tests,station
0,2026-01-01 00:00:00+00:00,37.7717,-122.235,0.0,-36.3,2,NaN,NaN
1,2026-01-01 00:06:00+00:00,37.7717,-122.235,0.0,-35.5,2,NaN,NaN
2,2026-01-01 00:12:00+00:00,37.7717,-122.235,0.0,-34.6,2,NaN,NaN
3,2026-01-01 00:18:00+00:00,37.7717,-122.235,0.0,-33.4,2,NaN,NaN
4,2026-01-01 00:24:00+00:00,37.7717,-122.235,0.0,-32.0,2,NaN,NaN
...,...,...,...,...,...,...,...,...
59756,2026-09-06 23:36:00+00:00,37.7717,-122.235,0.0,133.1,2,NaN,NaN
59757,2026-09-06 23:42:00+00:00,37.7717,-122.235,0.0,135.1,2,NaN,NaN
59758,2026-09-06 23:48:00+00:00,37.7717,-122.235,0.0,137.3,2,NaN,NaN
59759,2026-09-06 23:54:00+00:00,37.7717,-122.235,0.0,139.4,2,NaN,NaN


In [51]:
osk.find(featureType="timeSeries")

['ooi-gp02hypm-rim01-02-ctdmog039',
 'ooi-gp02hypm-wfp02-01-flordl000',
 'ooi-gp02hypm-wfp02-03-dostal000',
 'ooi-gp02hypm-wfp02-04-ctdpfl000',
 'ooi-gp02hypm-wfp02-05-vel3dl000',
 'ooi-gp02hypm-wfp03-01-flordl000',
 'ooi-gp02hypm-wfp03-03-dostal000',
 'ooi-gp02hypm-wfp03-04-ctdpfl000',
 'ooi-gp02hypm-wfp03-05-vel3dl000',
 'ooi-gp03flma-rim01-02-adcpsl003',
 'ooi-gp03flma-rim01-02-ctdmog040',
 'ooi-gp03flma-rim01-02-ctdmog041',
 'ooi-gp03flma-rim01-02-ctdmog042',
 'ooi-gp03flma-rim01-02-ctdmog043',
 'ooi-gp03flma-rim01-02-ctdmog044',
 'ooi-gp03flma-rim01-02-ctdmog045',
 'ooi-gp03flma-rim01-02-ctdmog046',
 'ooi-gp03flma-rim01-02-ctdmog047',
 'ooi-gp03flma-rim01-02-ctdmog048',
 'ooi-gp03flma-rim01-02-ctdmoh049',
 'ooi-gp03flma-rim01-02-ctdmoh050',
 'ooi-gp03flma-rim01-02-ctdmoh051',
 'ooi-gp03flma-ris01-03-dostad000',
 'ooi-gp03flma-ris01-04-phsenf000',
 'ooi-gp03flma-ris01-05-flortd000',
 'ooi-gp03flmb-rim01-02-adcpsl007',
 'ooi-gp03flmb-rim01-02-ctdmog060',
 'ooi-gp03flmb-rim01-02-ctdm

### See if two sources overlap

In [36]:
osk.overlap("ccs", "NOAA_DHW")

Overlap(space=yes, time=yes)

### Map locations from catalogs

In [40]:
osk.map_locations?

Signature:
osk.map_locations(
    what: 'Any' = None,
    *,
    catalog: 'str | None' = None,
    renderer: 'str' = 'matplotlib',
    domain: 'Any' = <object object at 0x776a3cc170d0>,
    **kwargs: 'Any',
)
Docstring:
Map where something sits: catalog datasets, or a plotted selection.

::

    osk.map_locations()                       # everything discoverable
    osk.map_locations(osk.find(variable="nitrate"))   # a catalog query result
    osk.find(variable="nitrate").map()         # the same, as a method
    osk.map_locations(comparison)              # where a Comparison's data sits
    osk.map_locations(comparison_set)          # a whole ComparisonSet
    osk.map_locations([comparison, "papa"])    # a mix of both

A catalog name (or ``None``, or a whole :func:`~ocean_skill.catalog.find`
result) draws from metadata alone, exactly as
:func:`~ocean_skill.plot.locations.build_items` always has — nothing is
read. A :class:`~ocean_skill.comparison.Comparison`,
:class:`~ocean_skill.comp

In [43]:
fig = osk.map_locations(catalog="OOI*", renderer="holoviews")
fig

:Overlay
   .WMTS.I            :WMTS   [Longitude,Latitude]
   .Points.TimeSeries :Points   [lon,lat]   (name,catalog,featureType,variables,time_coverage,cadence,resolution,depth,institution,title)

In [45]:
fig = osk.find(variable="nitrate").map(renderer="holoviews")
fig

/tmp/ipykernel_2108985/3789873409.py:1: UserWarning: skipping 1 source(s) with no declared geospatial extent: glodap — probe the catalog to fill extents in.
  fig = osk.find(variable="nitrate").map(renderer="holoviews")


:Overlay
   .WMTS.I            :WMTS   [Longitude,Latitude]
   .Rectangles.Grid   :Rectangles   [lon0,lat0,lon1,lat1]   (name,catalog,featureType,variables,time_coverage,cadence,resolution,depth,institution,title)
   .Points.TimeSeries :Points   [lon,lat]   (name,catalog,featureType,variables,time_coverage,cadence,resolution,depth,institution,title)